# Phase 2 - Legal NER with Arch 4 - DeepSeek-style MoE decoder

Starts from the Phase-1 model (12-layer MoE decoder (222.6M total, ~81M active): MLA attention, 32 routed + 1 shared expert per layer.) and fine-tunes it to tag the 12 legal entity types with
**exact character offsets** (strict macro-F1).

Five experiments run back to back, spread over both T4s; each one produces its own submission zip:

| Experiment | What is trained |
|---|---|
| `full` | every weight, with layer-wise learning-rate decay |
| `lorma` | **LoRMA** (compulsory): multiplicative low-rank update `(alpha/r * BA + I) W x` on the attention projections |
| `lora` | LoRA: additive low-rank update on the same projections |
| `partial` | only the top layers + final norm |
| `kd` | knowledge distillation: the fine-tuned `full` model teaches a half-depth student |

How the pieces fit:
* **Tagging**: BIO tags over tokens, decoded back to character spans. Punctuation is split one character per piece so
  gold boundaries line up with token boundaries (checked below - an "oracle" decode of the gold tags shows the ceiling).
* **Long documents** (avg. ~31K characters): 512-token windows with 128-token overlap; each token keeps the prediction
  from the window where it has the most context on both sides.
* **Decoder-only models** see the whole window during fine-tuning (causal mask lifted), following Dukic & Snajder (2024)
  *Looking Right is Sometimes Right* and BehnamGhader et al. (2024) *LLM2Vec*.
* **Decoding**: plain argmax, or a Viterbi pass that only allows valid BIO sequences; the better one on the dev
  documents is used for the test predictions.

References: Hu et al. 2021 (LoRA); Bihany, Patel & Modi 2025 (LoRMA); Hinton et al. 2015 (distillation);
Sanh et al. 2019 (DistilBERT layer selection); Howard & Ruder 2018 and Clark et al. 2020 (layer-wise LR decay).

**Kaggle settings:** GPU T4 x2, Internet on, secret `HF` holding your Hugging Face token. Then Run All.

## 1. Setup

In [9]:
import os, sys, json, glob, shutil, time
import torch
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

WORK_DIR = "/kaggle/working/phase2_arch4"
os.makedirs(WORK_DIR, exist_ok=True)
sys.path.insert(0, WORK_DIR)

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


GPUs: ['Tesla T4', 'Tesla T4']


## 2. Configuration
Everything you might want to change lives here.

In [10]:
%%writefile /kaggle/working/phase2_arch4/config.py
"""Every knob of this Phase-2 (legal NER) notebook."""
import os
import json

ARCH = "arch4"
WORK_DIR = f"/kaggle/working/phase2_{ARCH}"
DATA_DIR = os.path.join(WORK_DIR, "data")
CACHE_DIR = os.path.join(WORK_DIR, "phase1")        # Phase-1 files downloaded from Hugging Face land here
OUT_DIR = os.path.join(WORK_DIR, "runs")             # one folder per experiment

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
_P2 = os.path.join(DATA_DIR, "Phase_2_LNER")
FOLD_TEXT = [os.path.join(_P2, "Train", "fold1-text.json"), os.path.join(_P2, "Train", "fold2-text.json")]
FOLD_GOLD = [os.path.join(_P2, "Train", "fold1-gold.json"), os.path.join(_P2, "Train", "fold2-gold.json")]
TEST_TEXT = os.path.join(_P2, "Test", "fold3-text.json")

# ------------------------------------------------------------------ the Phase-1 model
# Looked up in this order: an attached Kaggle dataset whose path mentions "arch4", then these HF repos.
# The 600M-token run (v2) is preferred; the earlier 100M-token run is the fallback.
PHASE1_REPOS = ["D-240357/legal-arch4-moe-v2", "D-240357/legal-arch4-moe"]
WEIGHT_FILES = ["model_final.pt", "model_best.pt"]
TOKENIZER_FILES = ["tokenizer.json", "data/tokenizer.json"]
TOKENIZER_REPOS = PHASE1_REPOS + ["D-240357/legal-arch3-decoder"]
SPLIT_PUNCT = True             # one punctuation character per piece (see the alignment check in the notebook)
KAGGLE_HINT = "arch4"

NER_REPO = "D-240357/legal-ner-arch4"           # fine-tuned checkpoints, results and submission zips go here

# ------------------------------------------------------------------ experiments
# Each one is a separate submission. "kd" distils the fine-tuned KD_TEACHER into a half-depth student,
# so it waits for that run to finish.
METHODS = ["full", "lorma", "lora", "partial", "kd"]
KD_TEACHER = "full"

# ------------------------------------------------------------------ data
DEV_DOCS_PER_FOLD = 4          # 8 of the 70 training documents are held out to pick the best epoch / decoder
WINDOW = 512                   # tokens per window; documents average ~7K tokens
STRIDE = 384                   # 128-token overlap; every token is labelled from the window where it is most central
BIDIRECTIONAL = True            # lift the causal mask while fine-tuning (Dukic & Snajder 2024; LLM2Vec)
TRAIN_DOC_LIMIT = None         # set to e.g. 6 for a quick end-to-end smoke test

# ------------------------------------------------------------------ optimisation
EPOCHS = 10
PATIENCE = 3                   # stop after 3 epochs without a better dev F1
MICRO_BATCH = 4
GRAD_ACCUM = 4
EVAL_BATCH = 8
WARMUP_FRAC = 0.1
WEIGHT_DECAY = 0.01
HEAD_DROPOUT = 0.1
HEAD_LR = 1e-3

FULL_LR = 1e-4                 # top layer; each layer below gets LLRD times the one above it
LLRD = 0.85
PARTIAL_LR = 2e-4
PARTIAL_TOP_LAYERS = 4
ADAPTER_LR = 5e-4
ADAPTER_RANK = 8
LORA_ALPHA = 16
LORMA_ALPHA = 8
ADAPTER_DROPOUT = 0.05
ADAPTER_TARGETS = ["q_down", "q_up", "kv_down", "kv_up", "out"]       # attention projections, as in the LoRA and LoRMA papers
CLASS_WEIGHTING = "none"       # or "sqrt_inv" to up-weight rare tags

KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5                 # weight of the gold-label loss; the rest goes to matching the teacher
STUDENT_LAYERS = [0, 2, 4, 6, 8, 10]

SEED = 13

_override = os.environ.get("P2_CFG_OVERRIDE")
if _override:
    globals().update(json.loads(_override))

Overwriting /kaggle/working/phase2_arch4/config.py


## 3. Code
The Phase-1 tokenizer and model (unchanged, so the trained weights load), plus the NER pieces.

In [11]:
%%writefile /kaggle/working/phase2_arch4/tokenizer.py
"""
Byte-level BPE tokenizer, written from scratch (stdlib only for the tokenizer logic).

How it works, in short:
  1. A regex chops text into "pre-tokens" (a word with its leading space, a run of digits,
     a run of punctuation, ...). Merges never cross these boundaries.
  2. Every pre-token is turned into raw UTF-8 bytes (ids 0..255), so *any* text can be
     encoded - there is no <unk> and no lowercasing.
  3. Training repeatedly merges the most frequent adjacent id pair into a new id.
     We train on the table of *unique* pre-tokens with their counts, which is what keeps
     it fast in pure Python.
  4. Encoding replays the learned merges (lowest rank first) per pre-token, and caches the
     result, so repeated words cost a single dict lookup.
"""
import re
import json
import heapq
from collections import Counter, defaultdict

# Close to the GPT-2 splitter, but built on plain `re` (no \p{L}), so it needs no extra package.
PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+",
    re.UNICODE,
)

EOS_TOKEN = "<|endoftext|>"
PAD_TOKEN = "<|pad|>"


class ByteBPE:
    def __init__(self, merges=None, vocab_size=32000):
        self.vocab_size = vocab_size
        self.merges = [tuple(m) for m in (merges or [])]      # merge i creates id 256 + i
        self.eos_id = vocab_size - 2
        self.pad_id = vocab_size - 1
        self._cache = {}
        self._rebuild_tables()

    # ------------------------------------------------------------- internals
    def _rebuild_tables(self):
        self.ranks = {pair: (i, 256 + i) for i, pair in enumerate(self.merges)}
        vocab = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            vocab.append(vocab[a] + vocab[b])
        self.vocab = vocab
        self._cache.clear()

    # -------------------------------------------------------------- training
    @classmethod
    def train(cls, texts, vocab_size=32000, log_every=2000):
        """Learn merges from an iterable of strings."""
        n_merges = vocab_size - 256 - 2          # leave room for the two special tokens
        word_counts = Counter()
        for text in texts:
            word_counts.update(PRETOKEN_RE.findall(text))
        print(f"  unique pre-tokens: {len(word_counts):,}")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_count = defaultdict(int)
        where = defaultdict(set)                 # pair -> indices of words that (may) contain it
        for wi, w in enumerate(words):
            f = freqs[wi]
            for pair in zip(w, w[1:]):
                pair_count[pair] += f
                where[pair].add(wi)

        # Max-heap with lazy invalidation: stale entries are fixed up when they surface.
        heap = [(-c, p) for p, c in pair_count.items()]
        heapq.heapify(heap)

        merges = []
        vocab = [bytes([i]) for i in range(256)]
        for m in range(n_merges):
            best = None
            while heap:
                neg, pair = heapq.heappop(heap)
                real = pair_count.get(pair, 0)
                if real <= 0:
                    continue
                if real != -neg:
                    heapq.heappush(heap, (-real, pair))
                    continue
                best = pair
                break
            if best is None:
                print(f"  ran out of pairs after {m} merges")
                break

            new_id = 256 + m
            merges.append(best)
            a, b = best
            vocab.append(vocab[a] + vocab[b])
            touched = {}
            for wi in where.pop(best, ()):
                w = words[wi]
                f = freqs[wi]
                # find the occurrences in this word (left to right, non-overlapping)
                if len(w) < 2:
                    continue
                i, out, changed = 0, [], False
                while i < len(w):
                    if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                        changed = True
                    else:
                        out.append(w[i])
                        i += 1
                if not changed:
                    continue                      # stale entry in `where`
                for pair in zip(w, w[1:]):        # take the old word's pairs away ...
                    pair_count[pair] -= f
                for pair in zip(out, out[1:]):    # ... and add the new word's pairs
                    pair_count[pair] += f
                    where[pair].add(wi)
                    touched[pair] = pair_count[pair]
                words[wi] = out
            for pair, c in touched.items():
                if c > 0:
                    heapq.heappush(heap, (-c, pair))
            pair_count.pop(best, None)

            if (m + 1) % log_every == 0:
                print(f"  merge {m + 1:,}/{n_merges:,}  last = {vocab[-1]!r}")

        return cls(merges, vocab_size)

    # -------------------------------------------------------------- encoding
    def _encode_word(self, word):
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) > 1:
            best_rank, best_pair, best_new = None, None, None
            for pair in zip(ids, ids[1:]):
                hit = ranks.get(pair)
                if hit is not None and (best_rank is None or hit[0] < best_rank):
                    best_rank, best_pair, best_new = hit[0], pair, hit[1]
            if best_pair is None:
                break
            a, b = best_pair
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(best_new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out
        return ids

    def encode(self, text):
        cache = self._cache
        if len(cache) > 1_000_000:               # keep RAM bounded on huge corpora
            cache.clear()
        out = []
        for word in PRETOKEN_RE.findall(text):
            ids = cache.get(word)
            if ids is None:
                ids = cache[word] = self._encode_word(word)
            out.extend(ids)
        return out

    def decode(self, ids):
        specials = (self.eos_id, self.pad_id)
        data = b"".join(self.vocab[i] for i in ids if i < len(self.vocab) and i not in specials)
        return data.decode("utf-8", errors="replace")

    # ----------------------------------------------------------- persistence
    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges,
                       "eos": EOS_TOKEN, "pad": PAD_TOKEN}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            blob = json.load(f)
        return cls(blob["merges"], blob["vocab_size"])


# ---------------------------------------------------------------------------------------------------
# Phase 2 addition: encode with character offsets, so predicted tokens can be mapped back to exact spans.
# ---------------------------------------------------------------------------------------------------
# Same splitter as above, except a run of punctuation is cut into single characters. Gold entities often end
# in the middle of runs like "Ors.[1]" or "Cr.P.C.,"; with this split 99.3% of gold spans start and end on a
# token boundary (98.1% without it). Every piece is still encoded with the Phase-1 merges.
NER_PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]|\s+(?!\S)|\s+",
    re.UNICODE,
)


class NerTokenizer(ByteBPE):
    split_punct = True

    def encode_with_offsets(self, text):
        """Returns ids and, per token, the [start, end) of its non-whitespace characters in `text`."""
        splitter = NER_PRETOKEN_RE if self.split_punct else PRETOKEN_RE
        ids, starts, ends = [], [], []
        for m in splitter.finditer(text):
            word, base = m.group(), m.start()
            toks = self._cache.get(word)
            if toks is None:
                toks = self._cache[word] = self._encode_word(word)
            # byte position -> character position inside this piece (pieces can contain multi-byte characters)
            char_at_byte = []
            for ci, ch in enumerate(word):
                char_at_byte += [ci] * len(ch.encode("utf-8"))
            b = 0
            for t in toks:
                n = len(self.vocab[t])
                cs, ce = char_at_byte[b], char_at_byte[b + n - 1] + 1
                b += n
                piece = word[cs:ce]
                s = base + cs + (len(piece) - len(piece.lstrip()))
                e = base + ce - (len(piece) - len(piece.rstrip()))
                if s >= e:
                    s = e = base + ce
                ids.append(t)
                starts.append(s)
                ends.append(e)
        return ids, starts, ends

Overwriting /kaggle/working/phase2_arch4/tokenizer.py


In [12]:
%%writefile /kaggle/working/phase2_arch4/phase1_model.py
"""
Phase-1 model code, unchanged except for the small switches marked "Phase 2", so the saved weights load as-is.
"""
"""
Arch 4: a small DeepSeek-V3-flavoured mixture-of-experts decoder, written from scratch.

Ideas borrowed from the DeepSeek papers (scaled down to ~217M total / ~80M active parameters):

  * MLA - Multi-head Latent Attention. Keys/values are squeezed through a small latent vector,
    so the KV cache holds `kv_lora + qk_rope` numbers per token instead of full K and V.
    RoPE lives on a small separate slice of each head ("decoupled RoPE") so the latent stays
    position-free.
  * DeepSeekMoE - many small routed experts plus one shared expert that every token uses.
  * Auxiliary-loss-free load balancing (V3) - a per-expert bias is added to the routing scores
    *only to pick the top-k*; after every optimizer step it is nudged up for under-used experts
    and down for over-used ones. Gate weights come from the un-biased sigmoid scores. A tiny
    sequence-wise balance loss is kept only as a safety net.
  * Multi-token prediction (MTP) - during training an extra light module predicts the token two
    steps ahead. It shares the embedding / output head and is simply unused at inference.

Our own adaptation: expert dispatch uses fixed-size capacity buffers and one batched matmul over
all experts. Every tensor shape is static, so torch.compile can fuse it and there is no Python
loop over experts. In eval / generation the capacity is raised so that no token is ever dropped.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


@dataclass
class ModelConfig:
    vocab_size: int = 32000
    n_layer: int = 12
    n_head: int = 12
    d_model: int = 768
    max_seq_len: int = 2048
    rope_base: float = 10000.0
    norm_eps: float = 1e-5
    # MLA
    q_lora: int = 384
    kv_lora: int = 192
    qk_nope: int = 48
    qk_rope: int = 16
    v_dim: int = 64
    # MoE
    n_dense_layers: int = 1
    d_ff_dense: int = 2048
    n_routed: int = 32
    top_k: int = 4
    expert_ff: int = 192
    shared_ff: int = 512
    capacity_factor: float = 1.25
    # multi-token prediction
    mtp: bool = True
    mtp_ff: int = 1024


# ---------------------------------------------------------------------------
# small building blocks
# ---------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x32 * self.weight).to(x.dtype)


def build_rope_cache(dim, max_len, base):
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    angles = torch.outer(torch.arange(max_len).float(), inv_freq)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).to(x.dtype)


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.gate_up = nn.Linear(d_model, 2 * hidden, bias=False)
        self.down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        gate, up = self.gate_up(x).chunk(2, dim=-1)
        return self.down(F.silu(gate) * up)


# ---------------------------------------------------------------------------
# Multi-head Latent Attention
# ---------------------------------------------------------------------------
class MLA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.H = cfg.n_head
        self.nope, self.rope, self.vd = cfg.qk_nope, cfg.qk_rope, cfg.v_dim
        self.kv_lora = cfg.kv_lora

        self.q_down = nn.Linear(cfg.d_model, cfg.q_lora, bias=False)
        self.q_norm = RMSNorm(cfg.q_lora, cfg.norm_eps)
        self.q_up = nn.Linear(cfg.q_lora, self.H * (self.nope + self.rope), bias=False)

        # one projection yields the compressed KV latent AND the (shared) rotary key slice
        self.kv_down = nn.Linear(cfg.d_model, cfg.kv_lora + self.rope, bias=False)
        self.kv_norm = RMSNorm(cfg.kv_lora, cfg.norm_eps)
        self.kv_up = nn.Linear(cfg.kv_lora, self.H * (self.nope + self.vd), bias=False)

        self.out = nn.Linear(self.H * self.vd, cfg.d_model, bias=False)
        self.causal = True                         # Phase 2 turns this off to let tokens look right

    def forward(self, x, cos, sin, past=None, key_mask=None):
        B, T, _ = x.shape
        H = self.H

        q = self.q_up(self.q_norm(self.q_down(x))).view(B, T, H, self.nope + self.rope).transpose(1, 2)
        q_nope, q_rope = q.split([self.nope, self.rope], dim=-1)
        q_rope = apply_rope(q_rope, cos, sin)

        kv = self.kv_down(x)
        latent, k_rope = kv.split([self.kv_lora, self.rope], dim=-1)
        latent = self.kv_norm(latent)                                   # (B, T, kv_lora)
        k_rope = apply_rope(k_rope.unsqueeze(1), cos, sin)              # (B, 1, T, rope) shared by all heads

        if past is not None:                                            # cache = latent + rope key only
            latent = torch.cat([past[0], latent], dim=1)
            k_rope = torch.cat([past[1], k_rope], dim=2)
        present = (latent, k_rope)

        S = latent.size(1)
        kv_full = self.kv_up(latent).view(B, S, H, self.nope + self.vd).transpose(1, 2)
        k_nope, v = kv_full.split([self.nope, self.vd], dim=-1)

        q = torch.cat([q_nope, q_rope], dim=-1)
        k = torch.cat([k_nope, k_rope.expand(-1, H, -1, -1)], dim=-1)

        if key_mask is None:
            o = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal)
        else:
            o = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask)
        o = o.transpose(1, 2).reshape(B, T, H * self.vd)
        return self.out(o), present


# ---------------------------------------------------------------------------
# DeepSeekMoE layer with auxiliary-loss-free balancing and static-shape dispatch
# ---------------------------------------------------------------------------
@torch.compiler.disable
def _slot_positions(eid, E):
    """
    For every (token, expert) pair: its running position inside that expert's buffer, plus the
    one-hot assignment matrix. This is integer bookkeeping on ~33K elements, so it runs eagerly
    on purpose: comparing an index tensor with arange(E) trips an Inductor/Triton codegen bug on
    some torch builds (seen on Kaggle). Everything heavy around it stays compiled.
    """
    onehot = (eid.unsqueeze(1) == torch.arange(E, device=eid.device).unsqueeze(0)).to(torch.int32)
    pos = onehot.cumsum(0).gather(1, eid.unsqueeze(1)).squeeze(1) - 1
    return onehot, pos


class MoE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.E, self.K = cfg.n_routed, cfg.top_k
        self.cap_factor = cfg.capacity_factor
        self.dropless = False                      # Phase 2 fine-tuning routes every token
        D, h = cfg.d_model, cfg.expert_ff

        self.router = nn.Linear(D, self.E, bias=False)
        self.w_gate_up = nn.Parameter(torch.empty(self.E, D, 2 * h))    # all experts stacked -> one bmm
        self.w_down = nn.Parameter(torch.empty(self.E, h, D))
        self.shared = SwiGLU(D, cfg.shared_ff)

        nn.init.normal_(self.w_gate_up, std=0.02)
        nn.init.normal_(self.w_down, std=0.02)

        self.register_buffer("bias", torch.zeros(self.E))               # saved: it is learned state
        self.register_buffer("load", torch.zeros(self.E), persistent=False)
        self.register_buffer("dropped", torch.zeros(()), persistent=False)

    def forward(self, x):
        B, T, D = x.shape
        N, E, K = B * T, self.E, self.K
        flat = x.reshape(N, D)

        # --- routing: sigmoid affinities, biased top-k, normalised gate weights
        scores = torch.sigmoid(F.linear(flat.float(), self.router.weight.float()))      # (N, E)
        _, idx = torch.topk(scores.detach() + self.bias, K, dim=-1)                     # bias steers selection only
        gate = scores.gather(1, idx)
        gate = gate / (gate.sum(-1, keepdim=True) + 1e-9)

        # --- slot assignment: position of every (token, expert) pair inside its expert's buffer
        eid = idx.reshape(-1)                                                           # token-major, (N*K,)
        onehot, pos = _slot_positions(eid, E)

        if self.training and not self.dropless:
            cap = int(math.ceil(N * K / E * self.cap_factor / 8) * 8)
        else:
            cap = N                                                                     # drop-free for eval / generation
        keep = pos < cap
        slot = torch.where(keep, eid * cap + pos, torch.full_like(eid, E * cap))        # overflow -> spare row

        # --- dispatch -> one batched matmul over all experts -> combine
        dt = torch.get_autocast_dtype("cuda") if (torch.is_autocast_enabled() and x.is_cuda) else x.dtype
        xs = flat.to(dt).repeat_interleave(K, dim=0)
        buf = xs.new_zeros(E * cap + 1, D).index_put((slot,), xs)
        hid = buf[: E * cap].view(E, cap, D)
        g, u = torch.bmm(hid, self.w_gate_up.to(dt)).chunk(2, dim=-1)
        y = torch.bmm(F.silu(g) * u, self.w_down.to(dt))                                # (E, cap, D)
        y = torch.cat([y.reshape(E * cap, D), y.new_zeros(1, D)], dim=0)
        routed = (y[slot].view(N, K, D) * gate.to(y.dtype).unsqueeze(-1)).sum(1)

        out = (routed.to(x.dtype) + self.shared(flat)).view(B, T, D)

        if not self.training:
            return out, x.new_zeros((), dtype=torch.float32)

        # --- bookkeeping for balancing (no gradient) and the small sequence-wise safety loss
        with torch.no_grad():
            self.load += onehot.sum(0).float()
            self.dropped += (~keep).sum().float()
        p = (scores / (scores.sum(-1, keepdim=True) + 1e-9)).view(B, T, E).mean(1)      # mean affinity per sequence
        f = E * onehot.view(B, T * K, E).float().mean(1)                                # share of assignments per expert
        aux = (f * p).sum(-1).mean()
        return out, aux


# ---------------------------------------------------------------------------
# transformer block, MTP head, full model
# ---------------------------------------------------------------------------
class Block(nn.Module):
    def __init__(self, cfg, use_moe):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = MLA(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.use_moe = use_moe
        self.ffn = MoE(cfg) if use_moe else SwiGLU(cfg.d_model, cfg.d_ff_dense)

    def forward(self, x, cos, sin, past=None, key_mask=None):
        a, present = self.attn(self.attn_norm(x), cos, sin, past, key_mask)
        x = x + a
        if self.use_moe:
            f, aux = self.ffn(self.ffn_norm(x))
        else:
            f, aux = self.ffn(self.ffn_norm(x)), x.new_zeros((), dtype=torch.float32)
        return x + f, present, aux


class MTPHead(nn.Module):
    """Predicts token t+2 from (trunk hidden state at t, embedding of token t+1)."""

    def __init__(self, cfg):
        super().__init__()
        self.norm_h = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.norm_e = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.proj = nn.Linear(2 * cfg.d_model, cfg.d_model, bias=False)
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = MLA(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg.d_model, cfg.mtp_ff)

    def forward(self, h, next_emb, cos, sin):
        z = self.proj(torch.cat([self.norm_h(h), self.norm_e(next_emb)], dim=-1))
        a, _ = self.attn(self.attn_norm(z), cos, sin)
        z = z + a
        return z + self.ffn(self.ffn_norm(z))


class MoEGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg, use_moe=i >= cfg.n_dense_layers) for i in range(cfg.n_layer)])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mtp_head = MTPHead(cfg) if cfg.mtp else None

        cos, sin = build_rope_cache(cfg.qk_rope, cfg.max_seq_len, cfg.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        scale = 0.02 / math.sqrt(2 * cfg.n_layer)                       # GPT-2 style residual-branch scaling
        for blk in self.blocks:
            nn.init.normal_(blk.attn.out.weight, std=scale)
            if blk.use_moe:
                nn.init.normal_(blk.ffn.w_down, std=scale)
                nn.init.normal_(blk.ffn.shared.down.weight, std=scale)
            else:
                nn.init.normal_(blk.ffn.down.weight, std=scale)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    # ---- bookkeeping -------------------------------------------------------
    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def active_params(self):
        """Parameters touched per token: everything except the routed experts that were not picked."""
        total = self.num_params()
        routed = sum(b.ffn.w_gate_up.numel() + b.ffn.w_down.numel() for b in self.blocks if b.use_moe)
        inactive = routed * (1 - self.cfg.top_k / self.cfg.n_routed)
        mtp = sum(p.numel() for p in self.mtp_head.parameters()) if self.mtp_head is not None else 0
        return int(total - inactive - mtp)                              # MTP is training-only

    @torch.no_grad()
    def update_balance(self, speed):
        """
        Aux-loss-free balancing step (call once per optimizer step on every rank):
        push each expert's routing bias toward the mean load, then reset the counters.
        Returns (max load violation, fraction of assignments dropped) for logging.
        """
        moes = [b.ffn for b in self.blocks if b.use_moe]
        load = torch.stack([m.load for m in moes])
        dropped = torch.stack([m.dropped for m in moes])
        if dist.is_available() and dist.is_initialized():
            dist.all_reduce(load)
            dist.all_reduce(dropped)
        mean = load.mean(dim=-1, keepdim=True)
        if speed > 0:
            step = speed * torch.sign(mean - load)                      # under-used experts get a boost
            for m, s in zip(moes, step):
                m.bias += s
        maxvio = ((load - mean).abs().amax(dim=-1) / mean.squeeze(-1).clamp(min=1)).mean()
        drop_frac = dropped.sum() / load.sum().clamp(min=1)
        for m in moes:
            m.load.zero_()
            m.dropped.zero_()
        return maxvio, drop_frac

    def hidden_states(self, idx):
        """Final-layer hidden states (B, T, d_model) - what a Phase 2-7 task head sits on top of."""
        B, T = idx.shape
        cos = self.rope_cos[:T]
        sin = self.rope_sin[:T]
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x, _, _ = blk(x, cos, sin)
        return self.final_norm(x)

    # ---- forward -----------------------------------------------------------
    def forward(self, idx, targets=None, caches=None, start_pos=0, key_mask=None, last_only=False):
        """
        Training:   model(idx, targets) -> (ce, mtp_ce, seq_aux, z_loss)
        Inference:  model(idx, caches=..., ...) -> (logits, new_caches)
        """
        B, T = idx.shape
        cos = self.rope_cos[start_pos:start_pos + T]
        sin = self.rope_sin[start_pos:start_pos + T]

        x = self.tok_emb(idx)
        new_caches, aux_total = [], x.new_zeros((), dtype=torch.float32)
        for i, blk in enumerate(self.blocks):
            past = caches[i] if caches is not None else None
            x, present, aux = blk(x, cos, sin, past, key_mask)
            new_caches.append(present)
            aux_total = aux_total + aux

        hidden = x
        x = self.final_norm(x)
        if last_only:
            x = x[:, -1:, :]
        logits = F.linear(x, self.tok_emb.weight)

        if targets is None:
            return logits, new_caches

        V = logits.size(-1)
        logits = logits.float()
        ce = F.cross_entropy(logits.view(-1, V), targets.reshape(-1))
        z = torch.logsumexp(logits, dim=-1).pow(2).mean()

        mtp_ce = ce.new_zeros(())
        if self.mtp_head is not None and self.training:
            z2 = self.mtp_head(hidden, self.tok_emb(targets), cos, sin)         # embedding of token t+1
            lg2 = F.linear(self.final_norm(z2), self.tok_emb.weight).float()
            shifted = torch.cat([targets[:, 1:], torch.full_like(targets[:, :1], -100)], dim=1)   # token t+2
            mtp_ce = F.cross_entropy(lg2.view(-1, V), shifted.reshape(-1), ignore_index=-100)
        return ce, mtp_ce, aux_total / max(1, len(self.blocks) - self.cfg.n_dense_layers), z


# ---------------------------------------------------------------------------
# Generation (same batched, left-padded KV-cache sampler as arch 3)
# ---------------------------------------------------------------------------
def _sample(logits, temperature, top_k, top_p):
    logits = logits / max(temperature, 1e-5)
    k = min(top_k, logits.size(-1)) if top_k > 0 else logits.size(-1)
    vals, idx = torch.topk(logits, k, dim=-1)
    probs = F.softmax(vals, dim=-1)
    if top_p < 1.0:
        before = probs.cumsum(-1) - probs
        vals = vals.masked_fill(before >= top_p, float("-inf"))
        probs = F.softmax(vals, dim=-1)
    return idx.gather(-1, torch.multinomial(probs, 1)).squeeze(-1)


@torch.no_grad()
def generate(model, prompts, tokenizer, max_new_tokens=384, target_words=128,
             temperature=0.7, top_k=50, top_p=0.92, check_every=8, suppress_eos=True):
    """prompts: list of token-id lists. Returns the generated continuations as strings."""
    model.eval()
    dev = next(model.parameters()).device
    B = len(prompts)
    L = max(len(p) for p in prompts)
    assert L + max_new_tokens <= model.cfg.max_seq_len, "prompt + continuation exceeds context"
    pad, eos = tokenizer.pad_id, tokenizer.eos_id

    ids = torch.full((B, L), pad, dtype=torch.long)
    valid = torch.zeros((B, L), dtype=torch.bool)
    for r, p in enumerate(prompts):
        ids[r, L - len(p):] = torch.tensor(p, dtype=torch.long)
        valid[r, L - len(p):] = True
    ids, valid = ids.to(dev), valid.to(dev)

    causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=dev))
    mask = (causal[None, None] & valid[:, None, None, :]) | torch.eye(L, dtype=torch.bool, device=dev)[None, None]
    logits, caches = model(ids, caches=None, start_pos=0, key_mask=mask, last_only=True)

    out = torch.empty((B, 0), dtype=torch.long, device=dev)
    finished = torch.zeros(B, dtype=torch.bool, device=dev)
    for step in range(max_new_tokens):
        step_logits = logits[:, -1].float()
        step_logits[:, pad] = float("-inf")
        if suppress_eos:                          # the task wants a full-length continuation: stop on word count only
            step_logits[:, eos] = float("-inf")
        nxt = _sample(step_logits, temperature, top_k, top_p)
        nxt = torch.where(finished, torch.full_like(nxt, pad), nxt)
        out = torch.cat([out, nxt[:, None]], dim=1)
        finished |= nxt == eos

        if (step + 1) % check_every == 0:
            for r in torch.nonzero(~finished).flatten().tolist():
                if len(tokenizer.decode(out[r].tolist()).split()) >= target_words:
                    finished[r] = True
        if bool(finished.all()):
            break

        valid = torch.cat([valid, torch.ones((B, 1), dtype=torch.bool, device=dev)], dim=1)
        logits, caches = model(nxt[:, None], caches=caches, start_pos=L + step,
                               key_mask=valid[:, None, None, :], last_only=True)

    texts = []
    for r in range(B):
        toks = out[r].tolist()
        if eos in toks:
            toks = toks[:toks.index(eos)]
        toks = [t for t in toks if t != pad]
        texts.append(" ".join(tokenizer.decode(toks).split()[:target_words]))
    return texts

Overwriting /kaggle/working/phase2_arch4/phase1_model.py


In [13]:
%%writefile /kaggle/working/phase2_arch4/hub_files.py
"""Finds Phase-1 files: an attached Kaggle dataset first, then a local cache, then Hugging Face."""
import os
import re

from huggingface_hub import hf_hub_download

KAGGLE_INPUT = "/kaggle/input"


def _squash(text):
    return re.sub(r"[^a-z0-9]", "", text.lower())


def find_in_kaggle_input(filename, hint):
    """Only accepts a file whose path mentions the architecture (e.g. 'arch3', 'arch-3', 'Arch_3'),
    because arch 2, 3 and 4 all call their weights model_final.pt."""
    if not os.path.isdir(KAGGLE_INPUT):
        return None
    hits = []
    for root, _, files in os.walk(KAGGLE_INPUT):
        if filename in files:
            path = os.path.join(root, filename)
            if _squash(hint) in _squash(path):
                hits.append(path)
    return sorted(hits)[0] if hits else None


def fetch_from_hub(repo, filename, local_dir):
    try:
        return hf_hub_download(repo, filename, local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def locate(filenames, repos, hint, local_dir):
    """Names are tried in order of preference. Returns (path, where it came from)."""
    for name in filenames:
        path = find_in_kaggle_input(os.path.basename(name), hint)
        if path:
            return path, "kaggle input"
    for name in filenames:
        for repo in repos:
            cached = os.path.join(local_dir, _squash(repo), name)
            if os.path.exists(cached):
                return cached, f"cache of hf://{repo}"
            path = fetch_from_hub(repo, name, os.path.join(local_dir, _squash(repo)))
            if path:
                return path, f"hf://{repo}"
    raise FileNotFoundError(f"None of {filenames} found under /kaggle/input (path must mention '{hint}') or in "
                            f"{repos}. Attach the Phase-1 weights as a Kaggle dataset, or check HF_TOKEN and the repo names.")

Overwriting /kaggle/working/phase2_arch4/hub_files.py


In [14]:
%%writefile /kaggle/working/phase2_arch4/backbone.py
"""Loads the Phase-1 arch 4 weights (DeepSeek-style MoE decoder) and exposes them as a token encoder for NER."""
import os

import torch
import torch.nn as nn

import config as C
from hub_files import locate, fetch_from_hub  # noqa: F401  (fetch_from_hub is used by run_ner)
from phase1_model import MoEGPT, ModelConfig
from tokenizer import NerTokenizer


def load_tokenizer():
    path, _ = locate(C.TOKENIZER_FILES, C.TOKENIZER_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    tok = NerTokenizer.load(path)
    tok.split_punct = C.SPLIT_PUNCT
    return tok


class Backbone(nn.Module):
    def __init__(self, gpt, keep_layers, bidirectional):
        super().__init__()
        self.d_model = gpt.cfg.d_model
        self.tok_emb = gpt.tok_emb
        blocks = gpt.blocks if keep_layers is None else [gpt.blocks[i] for i in keep_layers]
        self.layers = nn.ModuleList(blocks)
        self.final_norm = gpt.final_norm
        self.register_buffer("rope_cos", gpt.rope_cos.clone(), persistent=False)
        self.register_buffer("rope_sin", gpt.rope_sin.clone(), persistent=False)
        self.bidirectional = bidirectional
        for blk in self.layers:
            blk.attn.causal = not bidirectional
            if blk.use_moe:
                blk.ffn.dropless = True        # never drop tokens while fine-tuning; routing biases stay frozen

    def forward(self, ids, mask):
        T = ids.size(1)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        key_mask = mask[:, None, None, :] if (self.bidirectional and not bool(mask.all())) else None
        x = self.tok_emb(ids)
        for blk in self.layers:
            x, _, _ = blk(x, cos, sin, None, key_mask)
        return self.final_norm(x)


def build_backbone(keep_layers=None, bidirectional=True):
    path, source = locate(C.WEIGHT_FILES, C.PHASE1_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    blob = torch.load(path, map_location="cpu")
    cfg = dict(blob["model_cfg"], mtp=False)               # the multi-token head is a pre-training-only extra
    gpt = MoEGPT(ModelConfig(**cfg))
    state = {k: v.float() for k, v in blob["model"].items() if not k.startswith("mtp_head.")}
    gpt.load_state_dict(state)
    if os.environ.get("P2_VERBOSE", "1") == "1":
        print(f"Phase-1 arch 4 weights: {path} ({source})", flush=True)
    return Backbone(gpt, keep_layers, bidirectional)


def student_layers():
    return list(C.STUDENT_LAYERS)

Overwriting /kaggle/working/phase2_arch4/backbone.py


In [15]:
%%writefile /kaggle/working/phase2_arch4/ner_core.py
"""
Phase 2 (legal NER) machinery shared by all four architectures:
data loading, BIO tagging, sliding windows, LoRA / LoRMA adapters, the tagging model,
training, span decoding and the strict (exact offsets + label) metric.
"""
import os
import json
import math
import random
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
TAGS = ["O"] + [f"{p}-{lab}" for lab in LABELS for p in ("B", "I")]
TAG_ID = {t: i for i, t in enumerate(TAGS)}
N_TAGS = len(TAGS)
IGNORE = -100


# --------------------------------------------------------------------------------------------- data
def load_docs(text_path, gold_path=None):
    with open(text_path, encoding="utf-8") as f:
        texts = json.load(f)
    gold = {}
    if gold_path:
        with open(gold_path, encoding="utf-8") as f:
            gold = json.load(f)
    docs = []
    for doc_id, text in texts.items():
        ents = [(int(e[0]), int(e[1]), e[2]) for e in gold.get(doc_id, [])]
        docs.append({"id": str(doc_id), "text": text, "ents": sorted(ents)})
    return docs


def split_dev(fold_docs, n_dev_per_fold, seed):
    """Hold out a few whole documents from every fold, so the dev set looks like the test set."""
    rng = random.Random(seed)
    train, dev = [], []
    for docs in fold_docs:
        docs = sorted(docs, key=lambda d: d["id"])
        rng.shuffle(docs)
        dev += docs[:n_dev_per_fold]
        train += docs[n_dev_per_fold:]
    return train, dev


# ------------------------------------------------------------------------------------ BIO encoding
def encode_doc(doc, tokenizer):
    """
    Tokenize with character offsets. starts/ends are the *content* span of every token (leading or
    trailing whitespace removed); pure-whitespace tokens get an empty span and never carry a label.
    """
    ids, starts, ends = tokenizer.encode_with_offsets(doc["text"])
    ids, starts, ends = np.asarray(ids, np.int64), np.asarray(starts, np.int64), np.asarray(ends, np.int64)
    real = ends > starts
    tags = np.where(real, TAG_ID["O"], IGNORE).astype(np.int64)

    for s, e, lab in doc["ents"]:
        hit = np.nonzero(real & (starts < e) & (ends > s))[0]
        if len(hit) == 0:
            continue
        tags[hit[0]] = TAG_ID[f"B-{lab}"]
        tags[hit[1:]] = TAG_ID[f"I-{lab}"]
    return {"id": doc["id"], "text": doc["text"], "ents": doc["ents"],
            "ids": ids, "starts": starts, "ends": ends, "tags": tags}


def make_windows(n, size, stride):
    if n <= size:
        return [(0, n)]
    wins = [(s, s + size) for s in range(0, n - size + 1, stride)]
    if wins[-1][1] < n:
        wins.append((n - size, n))
    return wins


def owner_of_tokens(windows, n):
    """For each token, the window in which it sits furthest from an edge (most context on both sides)."""
    best = np.full(n, -1, dtype=np.int64)
    best_margin = np.full(n, -1, dtype=np.int64)
    for k, (ws, we) in enumerate(windows):
        pos = np.arange(ws, we)
        margin = np.minimum(pos - ws, we - 1 - pos)
        better = margin > best_margin[ws:we]
        best[ws:we][better] = k
        best_margin[ws:we][better] = margin[better]
    return best


def window_items(enc_docs, size, stride):
    items = []
    for d, doc in enumerate(enc_docs):
        for ws, we in make_windows(len(doc["ids"]), size, stride):
            items.append((d, ws, we))
    return items


def collate(enc_docs, items, pad_id):
    T = max(we - ws for _, ws, we in items)
    B = len(items)
    ids = torch.full((B, T), pad_id, dtype=torch.long)
    tags = torch.full((B, T), IGNORE, dtype=torch.long)
    mask = torch.zeros((B, T), dtype=torch.bool)
    for r, (d, ws, we) in enumerate(items):
        doc = enc_docs[d]
        L = we - ws
        ids[r, :L] = torch.from_numpy(doc["ids"][ws:we])
        tags[r, :L] = torch.from_numpy(doc["tags"][ws:we])
        mask[r, :L] = True
    return ids, tags, mask


# ----------------------------------------------------------------------------------- adapters
class LoRALinear(nn.Module):
    """Hu et al. 2021: W x + (alpha / r) * B A x, with B = 0 at the start."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class LoRMALinear(nn.Module):
    """
    Bihany, Patel & Modi 2025 (LoRMA+): h = (alpha/r * B A + I) W0 x.
    Computed as W0x first, then A, then B, which costs the same as LoRA. B = 0 makes it the identity at the start.
    """

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        d = base.out_features
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.zeros(d, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + F.linear(F.linear(self.drop(h), self.A), self.B) * self.scale
        if self.base.bias is not None:
            h = h + self.base.bias
        return h


def inject_adapters(root, target_names, kind, r, alpha, dropout):
    wrapper = {"lora": LoRALinear, "lorma": LoRMALinear}[kind]
    swapped = 0
    for parent in list(root.modules()):
        for name, child in list(parent.named_children()):
            if name in target_names and isinstance(child, nn.Linear):
                setattr(parent, name, wrapper(child, r, alpha, dropout))
                swapped += 1
    return swapped


# --------------------------------------------------------------------------------------- model
class Tagger(nn.Module):
    def __init__(self, backbone, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(backbone.d_model, N_TAGS)
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, ids, mask):
        h = self.backbone(ids, mask)
        return self.head(self.drop(h.float()))


def configure_method(model, method, cfg):
    """Freeze / unfreeze / add adapters according to the experiment. Returns a short description."""
    bb = model.backbone
    if method in ("full", "kd"):
        for p in model.parameters():
            p.requires_grad = True
        return "all weights trainable"

    for p in bb.parameters():
        p.requires_grad = False

    if method == "partial":
        top = list(bb.layers)[-cfg.PARTIAL_TOP_LAYERS:]
        for layer in top:
            for p in layer.parameters():
                p.requires_grad = True
        for p in bb.final_norm.parameters():
            p.requires_grad = True
        return f"top {cfg.PARTIAL_TOP_LAYERS} layers + final norm trainable"

    if method in ("lora", "lorma"):
        n = inject_adapters(bb.layers, cfg.ADAPTER_TARGETS, method, cfg.ADAPTER_RANK,
                            cfg.LORA_ALPHA if method == "lora" else cfg.LORMA_ALPHA, cfg.ADAPTER_DROPOUT)
        return f"{n} linear layers wrapped with {method.upper()} (r={cfg.ADAPTER_RANK})"

    raise ValueError(f"unknown method {method}")


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, train


def trainable_state(model):
    """Only what training changed: adapters + head for PEFT runs, everything for full / KD."""
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().half() if v.is_floating_point() else v for k, v in model.state_dict().items() if k in names}


# ----------------------------------------------------------------------------------- optimizer
def layer_depth(name, n_layers):
    if ".layers." in name:
        return int(name.split(".layers.")[1].split(".")[0]) + 1
    if "emb" in name and name.startswith("backbone."):
        return 0
    return n_layers + 1                      # final norm and the head


def build_optimizer(model, method, cfg):
    n_layers = len(model.backbone.layers)
    groups = {}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if name.startswith("head."):
            lr = cfg.HEAD_LR
        elif method in ("full", "kd"):
            # layer-wise LR decay (ULMFiT / ELECTRA): lower layers move less
            lr = cfg.FULL_LR * cfg.LLRD ** (n_layers + 1 - layer_depth(name, n_layers))
        elif method == "partial":
            lr = cfg.PARTIAL_LR
        else:
            lr = cfg.ADAPTER_LR
        wd = 0.0 if p.ndim < 2 else cfg.WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()],
                             betas=(0.9, 0.98), eps=1e-6)


# ------------------------------------------------------------------------------------ decoding
def _allowed():
    allow = np.ones((N_TAGS, N_TAGS), dtype=bool)          # allow[prev, cur]
    for cur, tag in enumerate(TAGS):
        if tag.startswith("I-"):
            lab = tag[2:]
            allow[:, cur] = False
            allow[TAG_ID[f"B-{lab}"], cur] = True
            allow[TAG_ID[f"I-{lab}"], cur] = True
    start_ok = np.array([not t.startswith("I-") for t in TAGS])
    return allow, start_ok


ALLOW, START_OK = _allowed()


def viterbi(logp):
    """Best tag path that is a valid BIO sequence (no I-X without a preceding B-X / I-X)."""
    T = len(logp)
    if T == 0:
        return np.zeros(0, dtype=np.int64)
    trans = np.where(ALLOW, 0.0, -1e9)
    score = np.where(START_OK, logp[0], -1e9)
    back = np.zeros((T, N_TAGS), dtype=np.int64)
    for t in range(1, T):
        cand = score[:, None] + trans
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = np.zeros(T, dtype=np.int64)
    path[-1] = score.argmax()
    for t in range(T - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path


def greedy(logp):
    return logp.argmax(-1)


def tags_to_spans(tag_ids, starts, ends):
    """Turn a tag sequence over non-whitespace tokens into (start, end, label) character spans."""
    spans, cur = [], None
    for t, s, e in zip(tag_ids, starts, ends):
        tag = TAGS[t]
        if tag == "O":
            if cur:
                spans.append(tuple(cur))
            cur = None
            continue
        kind, lab = tag.split("-", 1)
        if kind == "B" or cur is None or cur[2] != lab:     # an orphan I- simply opens a new entity
            if cur:
                spans.append(tuple(cur))
            cur = [int(s), int(e), lab]
        else:
            cur[1] = int(e)
    if cur:
        spans.append(tuple(cur))
    return spans


# -------------------------------------------------------------------------------------- metric
def strict_scores(pred, gold):
    """pred / gold: {doc_id: iterable of (start, end, label)}. Exact span AND label must match."""
    tp, n_pred, n_gold = ({lab: 0 for lab in LABELS} for _ in range(3))
    for doc_id in gold:
        g, p = set(map(tuple, gold[doc_id])), set(map(tuple, pred.get(doc_id, [])))
        for s, e, lab in g:
            n_gold[lab] += 1
        for s, e, lab in p:
            n_pred[lab] += 1
            if (s, e, lab) in g:
                tp[lab] += 1
    per = {}
    for lab in LABELS:
        if n_gold[lab] == 0 and n_pred[lab] == 0:
            continue
        p = tp[lab] / n_pred[lab] if n_pred[lab] else 0.0
        r = tp[lab] / n_gold[lab] if n_gold[lab] else 0.0
        per[lab] = {"p": p, "r": r, "f1": 2 * p * r / (p + r) if p + r else 0.0, "gold": n_gold[lab]}
    macro = {k: float(np.mean([v[k] for v in per.values()])) if per else 0.0 for k in ("p", "r", "f1")}
    return macro, per


def oracle_spans(enc_doc):
    """Decode the gold tags themselves: shows the best score this tokenization allows."""
    real = enc_doc["ends"] > enc_doc["starts"]
    return tags_to_spans(enc_doc["tags"][real], enc_doc["starts"][real], enc_doc["ends"][real])


# ----------------------------------------------------------------------------------- inference
@torch.no_grad()
def doc_log_probs(model, enc_docs, cfg, device, pad_id, batch_size):
    """Per-token log-probabilities for whole documents, stitched from overlapping windows."""
    model.eval()
    out = []
    for doc in enc_docs:
        n = len(doc["ids"])
        wins = make_windows(n, cfg.WINDOW, cfg.STRIDE)
        owner = owner_of_tokens(wins, n)
        lp = np.zeros((n, N_TAGS), dtype=np.float32)
        items = [(0, ws, we) for ws, we in wins]
        for b in range(0, len(items), batch_size):
            chunk = items[b:b + batch_size]
            ids, _, mask = collate([doc], chunk, pad_id)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(ids.to(device), mask.to(device))
            logp = F.log_softmax(logits.float(), -1).cpu().numpy()
            for r, (_, ws, we) in enumerate(chunk):
                k = b + r
                own = np.nonzero(owner[ws:we] == k)[0]
                lp[ws + own] = logp[r, own]
        out.append(lp)
    return out


def decode_docs(enc_docs, log_probs, decoder):
    fn = viterbi if decoder == "viterbi" else greedy
    pred = {}
    for doc, lp in zip(enc_docs, log_probs):
        real = doc["ends"] > doc["starts"]
        tags = fn(lp[real])
        pred[doc["id"]] = tags_to_spans(tags, doc["starts"][real], doc["ends"][real])
    return pred


def evaluate(model, enc_docs, cfg, device, pad_id, batch_size):
    lps = doc_log_probs(model, enc_docs, cfg, device, pad_id, batch_size)
    gold = {d["id"]: d["ents"] for d in enc_docs}
    results = {}
    for dec in ("viterbi", "greedy"):
        macro, per = strict_scores(decode_docs(enc_docs, lps, dec), gold)
        results[dec] = {"macro": macro, "per_label": per}
    return results


def boundary_report(pred, gold):
    """Splits the errors into: exact, right label but wrong boundary, overlapping with the wrong label, missed."""
    counts = {"exact": 0, "boundary": 0, "wrong_label": 0, "missed": 0, "spurious": 0}
    for doc_id, g in gold.items():
        p = list(pred.get(doc_id, []))
        pset = set(p)
        for s, e, lab in g:
            if (s, e, lab) in pset:
                counts["exact"] += 1
                continue
            over = [q for q in p if q[0] < e and q[1] > s]
            if any(q[2] == lab for q in over):
                counts["boundary"] += 1
            elif over:
                counts["wrong_label"] += 1
            else:
                counts["missed"] += 1
        for s, e, lab in p:
            if not any(gs < e and ge > s for gs, ge, _ in g):
                counts["spurious"] += 1
    return counts


def write_submission(pred, test_docs, out_dir, zip_name):
    os.makedirs(out_dir, exist_ok=True)
    payload = {}
    for doc in test_docs:
        n = len(doc["text"])
        spans = sorted(set(pred.get(doc["id"], [])))
        assert all(0 <= s < e <= n and lab in LABELS for s, e, lab in spans), f"bad span in {doc['id']}"
        payload[doc["id"]] = [[s, e, lab] for s, e, lab in spans]
    pred_path = os.path.join(out_dir, "prediction.json")
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False)
    zip_path = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist() == ["prediction.json"]
    return pred_path, zip_path

Overwriting /kaggle/working/phase2_arch4/ner_core.py


In [16]:
%%writefile /kaggle/working/phase2_arch4/run_ner.py
"""
One Phase-2 experiment = one call:

    python run_ner.py --method lorma

Fine-tunes the Phase-1 model (or a distilled student of it) for legal NER, keeps the epoch with the best
dev strict macro-F1, writes the test predictions + submission zip and pushes everything to Hugging Face.
"""
import os
import sys
import json
import time
import random
import argparse

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as C
import backbone as BB
from ner_core import (load_docs, split_dev, encode_doc, window_items, collate, Tagger, configure_method,
                      count_params, trainable_state, build_optimizer, evaluate, doc_log_probs, decode_docs,
                      strict_scores, boundary_report, write_submission, N_TAGS, IGNORE)


def log(*a):
    print(*a, flush=True)


def build_tagger(method, keep_layers=None):
    model = Tagger(BB.build_backbone(keep_layers=keep_layers, bidirectional=C.BIDIRECTIONAL), C.HEAD_DROPOUT)
    note = configure_method(model, method, C)
    return model, note


def load_finetuned(method, device):
    """Rebuild a model trained earlier by this notebook (used as the KD teacher)."""
    path = os.path.join(C.OUT_DIR, method, f"ner_{method}.pt")
    if not os.path.exists(path):
        path = BB.fetch_from_hub(C.NER_REPO, f"{method}/ner_{method}.pt", os.path.join(C.OUT_DIR, "_hub"))
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"No fine-tuned '{method}' model found locally or on {C.NER_REPO}. "
                                f"Run the '{method}' experiment first - it is the distillation teacher.")
    blob = torch.load(path, map_location="cpu")
    model, _ = build_tagger(method, blob.get("keep_layers"))
    missing, unexpected = model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    assert not unexpected, unexpected
    return model.to(device).eval()


def class_weights(enc_docs, device):
    if C.CLASS_WEIGHTING != "sqrt_inv":
        return None
    counts = np.ones(N_TAGS)
    for d in enc_docs:
        t = d["tags"][d["tags"] != IGNORE]
        counts += np.bincount(t, minlength=N_TAGS)
    w = 1.0 / np.sqrt(counts / counts.sum())
    return torch.tensor(w / w.mean(), dtype=torch.float32, device=device)


def push(files, message):
    if not os.environ.get("HF_TOKEN"):
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(C.NER_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
        for local, remote in files:
            api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=C.NER_REPO,
                            token=os.environ["HF_TOKEN"], commit_message=message)
        log(f"pushed to {C.NER_REPO}: {message}")
    except Exception as e:
        log(f"HF push failed ({type(e).__name__}: {e}) - the files are still in {C.OUT_DIR}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True, choices=["full", "partial", "lora", "lorma", "kd"])
    args = ap.parse_args()
    method = args.method

    random.seed(C.SEED)
    np.random.seed(C.SEED)
    torch.manual_seed(C.SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = os.path.join(C.OUT_DIR, method)
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()

    # ---------------------------------------------------------------- data
    tok = BB.load_tokenizer()
    folds = [load_docs(C.FOLD_TEXT[i], C.FOLD_GOLD[i]) for i in range(2)]
    train_docs, dev_docs = split_dev(folds, C.DEV_DOCS_PER_FOLD, C.SEED)
    if C.TRAIN_DOC_LIMIT:                      # quick smoke test only
        train_docs, dev_docs = train_docs[:C.TRAIN_DOC_LIMIT], dev_docs[:max(1, C.TRAIN_DOC_LIMIT // 4)]
    test_docs = load_docs(C.TEST_TEXT)
    enc_train = [encode_doc(d, tok) for d in train_docs]
    enc_dev = [encode_doc(d, tok) for d in dev_docs]
    enc_test = [encode_doc(d, tok) for d in test_docs]
    log(f"[{method}] train docs {len(enc_train)} | dev docs {len(enc_dev)} | test docs {len(enc_test)} | "
        f"train tokens {sum(len(d['ids']) for d in enc_train):,}")

    # --------------------------------------------------------------- model
    keep = BB.student_layers() if method == "kd" else None
    model, note = build_tagger(method, keep)
    model.to(device)
    total, trainable = count_params(model)
    log(f"[{method}] {note} | params {total / 1e6:.1f}M, trainable {trainable / 1e6:.2f}M "
        f"({100 * trainable / total:.2f}%) | bidirectional attention: {C.BIDIRECTIONAL}")

    teacher = None
    if method == "kd":
        teacher = load_finetuned(C.KD_TEACHER, device)       # fp32 weights; autocast runs it in fp16 on the GPU
        t_total, _ = count_params(teacher)
        log(f"[kd] teacher = fine-tuned '{C.KD_TEACHER}' model ({t_total / 1e6:.1f}M) -> "
            f"student keeps layers {keep} ({total / 1e6:.1f}M)")

    # ------------------------------------------------------------ training
    items = window_items(enc_train, C.WINDOW, C.STRIDE)
    steps_per_epoch = math_ceil(len(items), C.MICRO_BATCH * C.GRAD_ACCUM)
    total_steps = steps_per_epoch * C.EPOCHS
    warmup = max(1, int(C.WARMUP_FRAC * total_steps))
    optimizer = build_optimizer(model, method, C)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    weights = class_weights(enc_train, device)
    rng = random.Random(C.SEED)

    best = {"f1": -1.0}
    stale, step = 0, 0
    history = []
    for epoch in range(1, C.EPOCHS + 1):
        model.train()
        rng.shuffle(items)
        running, seen = 0.0, 0
        for b in range(0, len(items), C.MICRO_BATCH):
            ids, tags, mask = collate(enc_train, items[b:b + C.MICRO_BATCH], tok.pad_id)
            ids, tags, mask = ids.to(device), tags.to(device), mask.to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(ids, mask).float()
                loss = F.cross_entropy(logits.view(-1, N_TAGS), tags.view(-1), weight=weights, ignore_index=IGNORE)
                if teacher is not None:
                    # Hinton et al. 2015: match the teacher's softened tag distribution on every real token
                    with torch.no_grad():
                        t_logits = teacher(ids, mask).float()
                    keep_tok = tags.view(-1) != IGNORE
                    T = C.KD_TEMPERATURE
                    kl = F.kl_div(F.log_softmax(logits.view(-1, N_TAGS)[keep_tok] / T, -1),
                                  F.softmax(t_logits.view(-1, N_TAGS)[keep_tok] / T, -1),
                                  reduction="batchmean") * T * T
                    loss = C.KD_ALPHA * loss + (1 - C.KD_ALPHA) * kl
            scaler.scale(loss / C.GRAD_ACCUM).backward()
            running += loss.item()
            seen += 1

            if seen % C.GRAD_ACCUM == 0 or b + C.MICRO_BATCH >= len(items):
                # linear warm-up, then linear decay to zero
                if step < warmup:
                    frac = (step + 1) / warmup
                else:
                    frac = max(0.0, (total_steps - step) / max(1, total_steps - warmup))
                for g, lr in zip(optimizer.param_groups, base_lrs):
                    g["lr"] = lr * frac
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                step += 1

        res = evaluate(model, enc_dev, C, device, tok.pad_id, C.EVAL_BATCH)
        dec = max(res, key=lambda k: res[k]["macro"]["f1"])
        f1 = res[dec]["macro"]["f1"]
        history.append({"epoch": epoch, "train_loss": running / max(1, seen),
                        "dev_f1_viterbi": res["viterbi"]["macro"]["f1"], "dev_f1_greedy": res["greedy"]["macro"]["f1"]})
        log(f"[{method}] epoch {epoch}/{C.EPOCHS} | loss {running / max(1, seen):.4f} | dev strict macro-F1 "
            f"viterbi {res['viterbi']['macro']['f1']:.4f}  greedy {res['greedy']['macro']['f1']:.4f} | "
            f"{(time.time() - t0) / 60:.1f} min")
        if f1 > best["f1"]:
            best = {"f1": f1, "epoch": epoch, "decoder": dec, "dev": res[dec]}
            torch.save({"method": method, "keep_layers": keep, "weights": trainable_state(model),
                        "dev_f1": f1, "decoder": dec, "epoch": epoch}, os.path.join(out_dir, f"ner_{method}.pt"))
            stale = 0
        else:
            stale += 1
            if stale >= C.PATIENCE:
                log(f"[{method}] no dev improvement for {C.PATIENCE} epochs - stopping early")
                break

    # ------------------------------------------------- best model -> test
    blob = torch.load(os.path.join(out_dir, f"ner_{method}.pt"), map_location="cpu")
    model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    t_inf = time.time()
    dev_lp = doc_log_probs(model, enc_dev, C, device, tok.pad_id, C.EVAL_BATCH)
    dev_pred = decode_docs(enc_dev, dev_lp, best["decoder"])
    dev_seconds = time.time() - t_inf
    errors = boundary_report(dev_pred, {d["id"]: d["ents"] for d in enc_dev})
    test_pred = decode_docs(enc_test, doc_log_probs(model, enc_test, C, device, tok.pad_id, C.EVAL_BATCH), best["decoder"])
    pred_path, zip_path = write_submission(test_pred, test_docs, out_dir, f"submission_{C.ARCH}_{method}.zip")

    summary = {
        "arch": C.ARCH, "method": method, "best_epoch": best["epoch"], "decoder": best["decoder"],
        "dev_macro": best["dev"]["macro"], "dev_per_label": best["dev"]["per_label"], "dev_errors": errors,
        "params_total": total, "params_trainable": trainable, "dev_inference_seconds": round(dev_seconds, 2),
        "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if device.type == "cuda" else 0.0,
        "train_minutes": round((time.time() - t0) / 60, 1), "history": history,
        "test_entities": sum(len(v) for v in test_pred.values()),
    }
    with open(os.path.join(out_dir, "results.json"), "w") as f:
        json.dump(summary, f, indent=2)
    log(f"[{method}] DONE | best dev strict macro-F1 {best['f1']:.4f} (epoch {best['epoch']}, {best['decoder']}) | "
        f"errors {errors} | {summary['test_entities']} test entities -> {zip_path}")

    push([(os.path.join(out_dir, f"ner_{method}.pt"), f"{method}/ner_{method}.pt"),
          (os.path.join(out_dir, "results.json"), f"{method}/results.json"),
          (zip_path, f"{method}/{os.path.basename(zip_path)}")], f"{C.ARCH} {method}: dev F1 {best['f1']:.4f}")


def math_ceil(a, b):
    return -(-a // b)


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/phase2_arch4/run_ner.py


## 4. Phase-2 data

In [17]:
import importlib
import config as C
importlib.reload(C)

for name in ["Train/fold1-text.json", "Train/fold1-gold.json", "Train/fold2-text.json",
             "Train/fold2-gold.json", "Train/labels.txt", "Test/fold3-text.json"]:
    hf_hub_download(C.HF_DATASET_REPO, f"Phase_2_LNER/{name}", repo_type="dataset", local_dir=C.DATA_DIR, token=HF_TOKEN)

import ner_core as N
importlib.reload(N)
folds = [N.load_docs(C.FOLD_TEXT[i], C.FOLD_GOLD[i]) for i in range(2)]
test_docs = N.load_docs(C.TEST_TEXT)
with open(os.path.join(C.DATA_DIR, "Phase_2_LNER/Train/labels.txt")) as f:
    assert f.read().split() == N.LABELS, "label list changed - update LABELS in ner_core.py"

n_ents = sum(len(d["ents"]) for fold in folds for d in fold)
print(f"train docs {sum(map(len, folds))} | test docs {len(test_docs)} | gold entities {n_ents}")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

train docs 70 | test docs 35 | gold entities 8393


## 5. Tokenizer alignment and offset self-test
Strict scoring needs exact character offsets. Here every training document is encoded, the gold entities are turned into
BIO tags, and those tags are decoded straight back into spans. The resulting "oracle" F1 is the best score this
tokenization allows, and it doubles as a test of the whole offset pipeline.

In [18]:
import backbone as BB
importlib.reload(BB)

tok = BB.load_tokenizer()
all_docs = [d for fold in folds for d in fold]
enc = [N.encode_doc(d, tok) for d in all_docs]
oracle = {d["id"]: N.oracle_spans(d) for d in enc}
macro, per = N.strict_scores(oracle, {d["id"]: d["ents"] for d in enc})
print(f"oracle strict macro-F1 = {macro['f1']:.4f}  (the ceiling for this tokenizer)")
print("per label:", {lab: round(v["f1"], 3) for lab, v in per.items()})

n_tok = [len(d["ids"]) for d in enc]
print(f"tokens per document: mean {sum(n_tok) / len(n_tok):,.0f}, max {max(n_tok):,}")
if hasattr(tok, "unk_id"):
    unk = sum(int((d["ids"] == tok.unk_id).sum()) for d in enc) / sum(n_tok)
    print(f"unknown-word tokens: {100 * unk:.1f}%")
assert macro["f1"] > 0.95, "offset pipeline looks broken - do not submit"

tokenizer.json:   0%|          | 0.00/413k [00:00<?, ?B/s]

oracle strict macro-F1 = 0.9945  (the ceiling for this tokenizer)
per label: {'APP': 0.985, 'RESP': 0.995, 'A.COUNSEL': 1.0, 'R.COUNSEL': 1.0, 'JUDGE': 1.0, 'WIT': 1.0, 'AUTH': 0.991, 'COURT': 0.982, 'STAT': 0.985, 'PREC': 0.996, 'DATE': 1.0, 'CASENO': 1.0}
tokens per document: mean 6,673, max 23,982


## 6. Load the Phase-1 weights once
This also caches them for the experiment processes.

In [19]:
probe = BB.build_backbone()
n = sum(p.numel() for p in probe.parameters())
ids = torch.tensor([enc[0]["ids"][:64].tolist()])
with torch.no_grad():
    h = probe(ids, torch.ones_like(ids, dtype=torch.bool))
print(f"backbone ready: {n / 1e6:.1f}M parameters, hidden states {tuple(h.shape)}")
del probe, h

model_best.pt: reconstructing file:   0%|          |  0.00B /  445MB            

model_best.pt: downloading bytes:           |  0.00B            

Phase-1 arch 4 weights: /kaggle/working/phase2_arch4/phase1/d240357legalarch4moev2/model_best.pt (hf://D-240357/legal-arch4-moe-v2)
backbone ready: 217.4M parameters, hidden states (1, 64, 768)


## 7. Run the experiments
Each experiment is its own process (`run_ner.py --method ...`); with two GPUs, two run at the same time.
`kd` starts once `full` (its teacher) has finished. Logs are prefixed with the GPU they run on.
Fine-tuned weights, results and submission zips are pushed to your NER repo on Hugging Face as they finish.

In [20]:
import subprocess, threading

pending = list(C.METHODS)
done, failed = set(), set()
lock = threading.Lock()


def next_job():
    with lock:
        if "kd" in pending and C.KD_TEACHER in failed:
            pending.remove("kd")
            print(f"skipping kd: its teacher '{C.KD_TEACHER}' failed")
        for m in pending:
            if m != "kd" or C.KD_TEACHER in done or C.KD_TEACHER not in C.METHODS:
                pending.remove(m)
                return m
        return "wait" if pending else None


def worker(gpu):
    while True:
        job = next_job()
        if job is None:
            return
        if job == "wait":
            time.sleep(15)
            continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        proc = subprocess.Popen([sys.executable, "run_ner.py", "--method", job], cwd=WORK_DIR, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(f"[gpu{gpu}] {line}", end="")
        with lock:
            (done if proc.wait() == 0 else failed).add(job)


t0 = time.time()
threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"\nfinished in {(time.time() - t0) / 60:.0f} min | ok: {sorted(done)} | failed: {sorted(failed)}")

[gpu1] [lorma] train docs 62 | dev docs 8 | test docs 35 | train tokens 423,748
[gpu0] [full] train docs 62 | dev docs 8 | test docs 35 | train tokens 423,748
[gpu1] Phase-1 arch 4 weights: /kaggle/working/phase2_arch4/phase1/d240357legalarch4moev2/model_best.pt (cache of hf://D-240357/legal-arch4-moe-v2)
[gpu0] Phase-1 arch 4 weights: /kaggle/working/phase2_arch4/phase1/d240357legalarch4moev2/model_best.pt (cache of hf://D-240357/legal-arch4-moe-v2)
[gpu0] [full] all weights trainable | params 217.5M, trainable 217.47M (100.00%) | bidirectional attention: True
[gpu1] [lorma] 60 linear layers wrapped with LORMA (r=8) | params 218.1M, trainable 0.69M (0.31%) | bidirectional attention: True
[gpu1] [lorma] epoch 1/10 | loss 1.0029 | dev strict macro-F1 viterbi 0.0499  greedy 0.0275 | 1.4 min
[gpu0] [full] epoch 1/10 | loss 0.7595 | dev strict macro-F1 viterbi 0.1709  greedy 0.1071 | 1.6 min
[gpu1] [lorma] epoch 2/10 | loss 0.1955 | dev strict macro-F1 viterbi 0.2669  greedy 0.1931 | 2.6 m

## 8. Results
Dev scores (strict, the competition metric), cost of each method, and where the errors are.

In [21]:
import pandas as pd

rows, per_label, errors = [], {}, {}
for path in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "results.json"))):
    r = json.load(open(path))
    m = r["method"]
    rows.append({"method": m, "dev macro-F1": r["dev_macro"]["f1"], "dev P": r["dev_macro"]["p"],
                 "dev R": r["dev_macro"]["r"], "decoder": r["decoder"], "best epoch": r["best_epoch"],
                 "params (M)": r["params_total"] / 1e6, "trainable %": 100 * r["params_trainable"] / r["params_total"],
                 "train min": r["train_minutes"], "peak GPU GB": r["peak_gpu_gb"], "dev infer s": r["dev_inference_seconds"]})
    per_label[m] = {lab: v["f1"] for lab, v in r["dev_per_label"].items()}
    errors[m] = r["dev_errors"]

table = pd.DataFrame(rows).sort_values("dev macro-F1", ascending=False).round(4)
display(table)
display(pd.DataFrame(per_label).round(3))
display(pd.DataFrame(errors).T)

,method,dev macro-F1,dev P,dev R,decoder,best epoch,params (M),trainable %,train min,peak GPU GB,dev infer s
2,lora,0.4630,0.4996,0.4804,viterbi,8,218.0790,0.2884,13.3,3.27,4.27
4,partial,0.4287,0.4343,0.4524,viterbi,2,217.4692,31.2036,5.1,3.20,3.97
0,full,0.4274,0.4639,0.4480,viterbi,6,217.4692,100.0000,14.0,7.41,4.05
1,kd,0.3790,0.4070,0.3820,viterbi,3,115.7119,100.0000,8.0,4.98,2.19
3,lorma,0.3689,0.4087,0.3576,viterbi,5,218.1358,0.3144,10.6,3.33,4.20


,full,kd,lora,lorma,partial
APP,0.094,0.169,0.047,0.073,0.072
RESP,0.125,0.182,0.200,0.114,0.129
A.COUNSEL,0.207,0.342,0.300,0.282,0.254
R.COUNSEL,0.430,0.105,0.467,0.254,0.306
JUDGE,0.392,0.209,0.269,0.273,0.326
AUTH,0.211,0.316,0.416,0.272,0.133
COURT,0.889,0.873,0.847,0.797,0.885
STAT,0.514,0.607,0.714,0.437,0.766
PREC,0.364,0.147,0.391,0.368,0.326
DATE,0.924,0.874,0.941,0.889,0.913


,exact,boundary,wrong_label,missed,spurious
full,548,133,75,292,103
kd,535,184,67,262,127
lora,614,143,63,228,100
lorma,494,188,54,312,98
partial,569,155,82,242,103


## 9. Pick a submission
Every experiment already wrote `runs/<method>/submission_<arch>_<method>.zip` (a single `prediction.json` at the
root, every test document id present). The best one on dev is copied to `/kaggle/working/submission.zip` too.
Submitting several of them counts as different methods.

In [22]:
best = table.iloc[0]["method"]
src = glob.glob(os.path.join(C.OUT_DIR, best, "submission_*.zip"))[0]
shutil.copy(src, "/kaggle/working/submission.zip")
print(f"best on dev: {best} -> /kaggle/working/submission.zip")
for z in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "submission_*.zip"))):
    print("  ", z)

best on dev: lora -> /kaggle/working/submission.zip
   /kaggle/working/phase2_arch4/runs/full/submission_arch4_full.zip
   /kaggle/working/phase2_arch4/runs/kd/submission_arch4_kd.zip
   /kaggle/working/phase2_arch4/runs/lora/submission_arch4_lora.zip
   /kaggle/working/phase2_arch4/runs/lorma/submission_arch4_lorma.zip
   /kaggle/working/phase2_arch4/runs/partial/submission_arch4_partial.zip
